# Параметрическое сканирование коэффициента заразности β

**Цель:** Исследовать, как изменение базовой заразности (β_und и пропорционально β_det)
влияет на эпидемические показатели: пик заболеваемости, долю переболевших и число умерших.

Выполняется параметрическое сканирование с несколькими повторными прогонами для учёта
стохастичности модели.

## Инициализация проекта и загрузка пакетов

In [ ]:
using DrWatson
@quickactivate "project"
using Agents, DataFrames, Plots, CSV, Random

Подключение модуля с определением модели SIR

In [ ]:
include(srcdir("sir_model.jl"))

## Функция запуска одного эксперимента

`run_experiment(p)` — запускает симуляцию с заданными параметрами и возвращает
ключевые метрики эпидемии:
- `peak` — пиковая доля инфицированных
- `final_inf` — конечная доля инфицированных
- `final_rec` — конечная доля выздоровевших
- `deaths` — общее число умерших

In [ ]:
function run_experiment(p)

### Создание параметров заразности
На основе скалярного значения `beta` создаём векторы для трёх городов:
- β_und — заразность невыявленных (основной параметр)
- β_det — заразность выявленных (в 10 раз ниже)

In [ ]:
    beta = p[:beta]
    β_und = fill(beta, 3)
    β_det = fill(beta/10, 3)

### Инициализация модели
Передаём все параметры в функцию создания модели

In [ ]:
    model = initialize_sir(;
        Ns = p[:Ns],
        β_und = β_und,
        β_det = β_det,
        infection_period = p[:infection_period],
        detection_time = p[:detection_time],
        death_rate = p[:death_rate],
        reinfection_probability = p[:reinfection_probability],
        Is = p[:Is],
        seed = p[:seed],
        n_steps = p[:n_steps],
    )

### Вспомогательная функция
Вычисляет долю инфицированных в текущий момент времени

In [ ]:
    infected_fraction(model) = count(a.status == :I for a in allagents(model)) / nagents(model)

### Основной цикл симуляции
Выполняем ручной шаг для каждого агента (более безопасный подход)

In [ ]:
    peak_infected = 0.0

    for step = 1:p[:n_steps]

Обход всех агентов с безопасной обработкой удалённых

In [ ]:
        agent_ids = collect(allids(model))
        for id in agent_ids
            agent = try
                model[id]
            catch
                nothing
            end
            if agent !== nothing
                sir_agent_step!(agent, model)
            end
        end

Обновляем пиковое значение

In [ ]:
        frac = infected_fraction(model)
        if frac > peak_infected
            peak_infected = frac
        end
    end

### Сбор финальных метрик
- `final_infected` — доля инфицированных в конце симуляции
- `final_recovered` — доля выздоровевших
- `total_deaths` — общее число умерших

In [ ]:
    final_infected = infected_fraction(model)
    final_recovered = count(a.status == :R for a in allagents(model)) / nagents(model)
    total_deaths = sum(p[:Ns]) - nagents(model)

    return (
        peak = peak_infected,
        final_inf = final_infected,
        final_rec = final_recovered,
        deaths = total_deaths,
    )
end

## Формирование сетки параметров

Исследуем диапазон коэффициента заразности β от 0.1 до 1.0 с шагом 0.1.
Для каждого значения β выполняем 3 прогона с разными случайными зёрнами
для учёта стохастичности.

Диапазон значений β (коэффициент заразности)

In [ ]:
beta_range = 0.1:0.1:1.0

Набор случайных зёрен для воспроизводимости

In [ ]:
seeds = [42, 43, 44]

### Создание списка параметров

Генерируем все комбинации (β, seed) для последующего запуска.

In [ ]:
params_list = []

for b in beta_range
    for s in seeds
        push!(
            params_list,
            Dict(
                :beta => b,                              # коэффициент заразности
                :Ns => [1000, 1000, 1000],               # численность населения в городах
                :infection_period => 14,                 # длительность болезни (дней)
                :detection_time => 7,                    # время до выявления (дней)
                :death_rate => 0.02,                     # вероятность смерти
                :reinfection_probability => 0.1,         # вероятность повторного заражения
                :Is => [0, 0, 1],                        # начальные заражённые (только в городе 3)
                :seed => s,                              # зерно случайных чисел
                :n_steps => 100,                         # длительность симуляции (дней)
            ),
        )
    end
end

## Запуск экспериментов

Последовательно выполняем симуляции для всех комбинаций параметров
и собираем результаты.

In [ ]:
results = []

for params in params_list
    data = run_experiment(params)
    push!(results, merge(params, Dict(pairs(data))))
    println("Завершён эксперимент с beta = $(params[:beta]), seed = $(params[:seed])")
end

## Сохранение результатов

Сохраняем данные всех прогонов в CSV-файл для последующего анализа
и возможного воспроизведения.

In [ ]:
df = DataFrame(results)
CSV.write(datadir("beta_scan_all.csv"), df)

## Усреднение по повторным прогонам

Группируем данные по значению β и усредняем метрики по трём случайным зёрнам.
Это позволяет получить более устойчивые оценки.

In [ ]:
using Statistics

grouped = combine(
    groupby(df, [:beta]),
    :peak => mean => :mean_peak,
    :final_inf => mean => :mean_final_inf,
    :deaths => mean => :mean_deaths,
)

## Визуализация результатов

Строим график зависимости эпидемических показателей от коэффициента заразности β:
- **Пик эпидемии** — максимальная доля инфицированных
- **Конечная доля инфицированных** — доля переболевших к концу эпидемии
- **Доля умерших** — нормированная на общую численность (3000 человек)

In [ ]:
plot(
    grouped.beta,
    grouped.mean_peak,
    label = "Пик эпидемии",
    xlabel = "Коэффициент заразности β",
    ylabel = "Доля инфицированных",
    marker = :circle,
    linewidth = 2,
)

plot!(
    grouped.beta,
    grouped.mean_final_inf,
    label = "Конечная доля инфицированных",
    marker = :square,
)

plot!(
    grouped.beta,
    grouped.mean_deaths ./ 3000,
    label = "Доля умерших",
    marker = :diamond,
)

Сохраняем график

In [ ]:
savefig(plotsdir("beta_scan.png"))

## Вывод информации о результатах

In [ ]:
println("Результаты сохранены в data/beta_scan_all.csv и plots/beta_scan.png")

## Интерпретация результатов

Полученный график позволяет:

1. **Найти пороговое значение β** — минимальный коэффициент заразности,
   при котором возникает эпидемия (пик > 5% популяции)

2. **Оценить рост нагрузки** на систему здравоохранения — как увеличивается
   пиковая заболеваемость с ростом β

3. **Сравнить теоретический порог R₀ = 1** с наблюдаемым поведением модели

4. **Проанализировать зависимость смертности** от заразности инфекции

Типичные выводы:
- При низких β (0.1–0.2) эпидемия не развивается
- При β ≈ 0.3–0.4 достигается порог распространения
- Дальнейший рост β приводит к увеличению всех показателей
- Доля умерших растёт нелинейно из-за эффекта перегрузки системы